In [0]:
from pyspark.sql.functions import col, min as spark_min, unix_timestamp

catalog = "transit_analytics"
silver_schema = "silver"
gold_schema = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")

In [0]:
from pyspark.sql.functions import col, min as spark_min, unix_timestamp

vp = spark.table(f"{catalog}.{silver_schema}.vehicle_positions")
tu = spark.table(f"{catalog}.{silver_schema}.trip_updates")

# For each trip on a given service day, find the single NEXT upcoming stop
# (earliest arrival_ts still ahead). Grouping by start_date too prevents
# matching a vehicle reading to a stop-time prediction from a different day
# when trip_id values get reused across service days.
next_stop = (
    tu.groupBy("trip_id", "start_date")
    .agg(spark_min("arrival_ts").alias("next_stop_arrival_ts"))
)

gold_vtf_df = (
    vp.join(next_stop, on=["trip_id", "start_date"], how="inner")
    .withColumn(
        "seconds_to_next_stop",
        unix_timestamp(col("next_stop_arrival_ts")) - unix_timestamp(col("vehicle_ts"))
    )
    .filter(col("seconds_to_next_stop") > 0)
    .filter(col("seconds_to_next_stop") < 1800)   # sanity cap: no real "next stop" should be 30+ min away
    .select(
        "trip_id", "route_id", "vehicle_id",
        "latitude", "longitude", "speed", "bearing",
        "vehicle_ts", "next_stop_arrival_ts", "seconds_to_next_stop"
    )
)

gold_vtf_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{gold_schema}.vehicle_trip_features"
)

print("vehicle_trip_features row count:", gold_vtf_df.count())
display(gold_vtf_df.limit(10))

In [0]:
from pyspark.sql.functions import avg, min as spark_min, max as spark_max, count

gold_route_summary_df = (
    vp.groupBy("route_id")
    .agg(
        avg("speed").alias("avg_speed"),
        spark_min("speed").alias("min_speed"),
        spark_max("speed").alias("max_speed"),
        count("*").alias("reading_count"),
    )
    .orderBy(col("reading_count").desc())
)

gold_route_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{gold_schema}.route_performance_summary"
)

print("route_performance_summary row count:", gold_route_summary_df.count())
display(gold_route_summary_df)